# openEO NDWI map

It creates a **preview of NDWI from temporal median reflectance**, rather than parcel statistics. It omits local IQR cleaning and gap filling. The original date range and extent are retained; the 1 km preview grid keeps the downloaded map manageable. For parcel-scale detail, select a small extent before reducing the resolution to 10 m.

References: [pipeline workflow](../../docs/data_preparation/openeo/WORKFLOW.md), [database setup](../../docs/data_preparation/openeo/CREDENTIALS.md), and [JupyterGIS API](https://jupytergis.readthedocs.io/en/latest/user_guide/python_api/api.html).

In [ ]:
import base64
import hashlib
import json
import time
from datetime import date, timedelta
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openeo
import rioxarray
from IPython.display import display
from jupytergis import GISDocument
from PIL import Image
from openeo.processes import if_

from data_preparation.parcel_stats import OpenEOZonalStats
from data_preparation.parcel_stats.multiuser import load_openeo_users_from_db

## Configuration

`END_DATE` is inclusive. The request adds one day for openEO's exclusive upper boundary. Outputs go into a dedicated directory, with filenames derived from the process graph to avoid reusing a different request.

In [ ]:
START_DATE = "2025-07-01"
END_DATE = "2025-08-31"
spatial_extent = {
    "west": 22,
    "south": 39.5,
    "east": 22.2,
    "north": 39.7,
    "crs": "EPSG:4326",
}
MAP_RESOLUTION_METRES = 1000
MAP_PROJECTION = "EPSG:3857"
OUTPUT_DIR = Path("C:/work_dir/satellites/outputs/experiments/openeo_map")
ACCOUNT_INDEX = 1

if date.fromisoformat(START_DATE) > date.fromisoformat(END_DATE):
    raise ValueError("Error: START_DATE must not be after END_DATE.")
if MAP_RESOLUTION_METRES <= 0:
    raise ValueError("Error: MAP_RESOLUTION_METRES must be positive.")
if not (-180 <= spatial_extent["west"] < spatial_extent["east"] <= 180
        and -85 < spatial_extent["south"] < spatial_extent["north"] < 85):
    raise ValueError("Error: Choose ordered WGS84 bounds within the Web Mercator latitude range.")
temporal_extent = [START_DATE, (date.fromisoformat(END_DATE) + timedelta(days=1)).isoformat()]

## Connect with the pipeline's database accounts

Set `DB_NAME`, `TBL_USERS`, and `POSTGRES_*` in the kernel environment as described in the database guide. Credentials stay in memory; do not print the account tuples.

In [ ]:
users_list = load_openeo_users_from_db()
if not 0 <= ACCOUNT_INDEX < len(users_list):
    raise ValueError("Error: ACCOUNT_INDEX is outside the available account list.")
print(f"User selected: {users_list[ACCOUNT_INDEX][0]}")

connection = openeo.connect(OpenEOZonalStats.OPENEO_URL)
connection.authenticate_oidc_resource_owner_password_credentials(
    username=users_list[ACCOUNT_INDEX][0],
    password=users_list[ACCOUNT_INDEX][1],
    client_id=OpenEOZonalStats.OPENEO_OIDC_PASSWORD_CLIENT_ID,
)
print("Connected to the CDSE openEO backend.")

## Build the cloud-masked NDWI graph

Align SCL with the 10 m reflectance grid using nearest-neighbor resampling. Apply the pipeline's invalid classes and one-pixel cloud buffer before reducing the spatial resolution. The preview then takes the temporal median of each physical band and calculates `(B03 - B08) / (B03 + B08)`, masking zero denominators. This is not the median of per-acquisition NDWI.

Graph construction does not start processing. The next section submits a remote batch job; a year over the retained large extent can still take significant time despite the coarse output grid.

In [ ]:
cube = connection.load_collection(
    OpenEOZonalStats.SENTINEL2_COLLECTION,
    bands=["B03", "B08", OpenEOZonalStats.SENTINEL2_SCENE_CLASSIFICATION_BAND],
    temporal_extent=temporal_extent,
    spatial_extent=spatial_extent,
    max_cloud_cover=OpenEOZonalStats.SENTINEL2_MAX_SCENE_CLOUD_COVER,
)
reflectance = cube.filter_bands(["B03", "B08"]).resample_spatial(
    resolution=OpenEOZonalStats.TARGET_RESOLUTION_METRES,
    projection=MAP_PROJECTION,
    method="bilinear",
) * OpenEOZonalStats.SENTINEL2_REFLECTANCE_SCALE_FACTOR
scl = cube.band(OpenEOZonalStats.SENTINEL2_SCENE_CLASSIFICATION_BAND).resample_cube_spatial(
    reflectance, method="near"
)
invalid = scl == OpenEOZonalStats.SENTINEL2_INVALID_SCL_CLASSES[0]
for class_id in OpenEOZonalStats.SENTINEL2_INVALID_SCL_CLASSES[1:]:
    invalid = invalid | (scl == class_id)
cloud = scl == OpenEOZonalStats.SENTINEL2_BUFFERED_SCL_CLASSES[0]
for class_id in OpenEOZonalStats.SENTINEL2_BUFFERED_SCL_CLASSES[1:]:
    cloud = cloud | (scl == class_id)
cloud_buffer = cloud.apply_kernel([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) > 0
masked = reflectance.mask(invalid | cloud_buffer)
preview = masked.resample_spatial(
    resolution=MAP_RESOLUTION_METRES, projection=MAP_PROJECTION, method="average"
)
composite = preview.reduce_dimension(dimension="t", reducer="median")
# The band order is explicitly B03, B08. Null denominators stay null.
ndwi = composite.reduce_dimension(
    dimension="bands",
    reducer=lambda bands: (bands[0] - bands[1]) / if_(
        bands[0] + bands[1] != 0, bands[0] + bands[1], None
    ),
)
print("NDWI process graph built.")

## Execute and download

The job ID is saved before starting, so rerunning this cell resumes the same job with the same account. Completed GeoTIFFs are reused; partial downloads are not. If a remote job fails, inspect its logs before removing its job-ID file to deliberately submit a replacement.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
graph_json = json.dumps(ndwi.flat_graph(), sort_keys=True)
request_id = hashlib.sha256(graph_json.encode("utf-8")).hexdigest()[:16]
raster_path = OUTPUT_DIR / f"ndwi_{request_id}.tif"
job_path = OUTPUT_DIR / f"ndwi_{request_id}.job-id.txt"
partial_path = raster_path.with_suffix(".partial.tif")

if not raster_path.is_file():
    if job_path.is_file():
        job = connection.job(job_path.read_text(encoding="utf-8").strip())
    else:
        job = ndwi.create_job(out_format="GTiff", title=f"NDWI map {START_DATE} to {END_DATE}")
        job_path.write_text(job.job_id, encoding="utf-8")
    print(f"Remote job: {job.job_id}")
    status = job.status()
    if status == "created":
        job.start()
    while status in {"created", "submitted", "queued", "running"}:
        time.sleep(15)
        status = job.status()
        print(f"Job status: {status}")
    if status != "finished":
        raise RuntimeError(f"Error: Job {job.job_id} ended with status {status}; inspect job.logs().")
    job.get_results().download_file(partial_path)
    partial_path.replace(raster_path)
print(f"GeoTIFF: {raster_path}")

## Load and colour the preview

The GeoTIFF retains numeric NDWI values. The PNG is only a coloured display, with missing pixels transparent and a fixed scale from -1 to 1. JupyterGIS receives a georeferenced image rather than an openEO graph or Xarray object, so this works without `jupyter-tiler` (unavailable for Python 3.14).

In [ ]:
with rioxarray.open_rasterio(raster_path, masked=True) as raster:
    if raster.sizes["band"] != 1 or raster.rio.crs is None:
        raise ValueError("Error: Expected a single-band NDWI GeoTIFF with a CRS.")
    ndwi_array = raster.isel(band=0, drop=True).load()
if ndwi_array.rio.crs.to_epsg() != 3857:
    ndwi_array = ndwi_array.rio.reproject(MAP_PROJECTION)
# PNG rows run north to south; use the same orientation as the map.
ndwi_array = ndwi_array.sortby("y", ascending=False).sortby("x")
values = ndwi_array.values
valid = np.isfinite(values)
if not valid.any():
    raise ValueError("Error: No valid NDWI pixels; check the extent, dates, and cloud coverage.")
rgba = plt.colormaps["viridis"](np.clip((np.nan_to_num(values) + 1) / 2, 0, 1), bytes=True)
rgba[~valid, 3] = 0
png_buffer = BytesIO()
Image.fromarray(rgba).save(png_buffer, format="PNG")
image_url = "data:image/png;base64," + base64.b64encode(png_buffer.getvalue()).decode("ascii")
west, south, east, north = ndwi_array.rio.transform_bounds("EPSG:4326")
coordinates = [[west, north], [east, north], [east, south], [west, south]]

fig, ax = plt.subplots(figsize=(8, 3))
preview_image = ax.imshow(values, cmap="viridis", vmin=-1, vmax=1)
ax.set_title(f"NDWI of median reflectance | {START_DATE} to {END_DATE}")
ax.set_axis_off()
fig.colorbar(preview_image, ax=ax, label="NDWI", shrink=0.8)
plt.show()

## Interactive map

Display the widget before waiting for frontend readiness. This requires a frontend that supports JupyterGIS widgets; if VS Code does not render it, open this notebook in JupyterLab with JupyterGIS enabled. The static preview above remains available.

In [ ]:
doc = GISDocument(
    latitude=(south + north) / 2,
    longitude=(west + east) / 2,
    zoom=6,
    projection=MAP_PROJECTION,
)
display(doc)

In [ ]:
await doc.ready()
doc.add_image_layer(
    url=image_url,
    coordinates=coordinates,
    name="NDWI (-1 to 1, viridis)",
    zoom_to=True,
)

## Optional test: direct Xarray NDWI layer

[`GISDocument.add_data_array_layer()`](https://jupytergis.readthedocs.io/en/latest/user_guide/python_api/api.html#jupytergis.GISDocument.add_data_array_layer) accepts an **Xarray DataArray**. Use `ndwi_array` loaded above, which contains numeric NDWI and its CRS; `ndwi` is still a remote openEO process graph and cannot be passed to this method.

Run these cells after **Load and colour the preview**. They create a separate map with the same `viridis` colour scale, without submitting another openEO job or converting the array to PNG. The image-layer map is optional for this test.

This method requires `jupyter-tiler`. The earlier installation check found that available releases require Python >=3.12,<3.14, whereas `eo_spatial` uses Python 3.14. The check below reports whether the dependency can now be imported and skips this optional map if it cannot. To test with those releases, use a separate Python 3.12/3.13 environment with the project dependencies and `jupytergis[tiler]` installed, then select that kernel. Keep the kernel and JupyterGIS frontend compatible. Restart the kernel after installing dependencies.

In [ ]:
import sys

try:
    import jupyter_tiler.titiler
except ImportError as exc:
    data_array_layer_available = False
    print(f"Direct DataArray test unavailable in Python {sys.version.split()[0]}: {exc}")
    print("Use a compatible environment with jupytergis[tiler] installed, then rerun this section.")
else:
    data_array_layer_available = True
    print("jupyter-tiler is available; continue with the direct NDWI DataArray map.")

In [ ]:
if data_array_layer_available:
    if "ndwi_array" not in globals():
        raise RuntimeError("Error: Run the GeoTIFF loading cell to create ndwi_array first.")
    if ndwi_array.ndim != 2 or ndwi_array.rio.crs is None:
        raise ValueError("Error: The NDWI DataArray must have two spatial dimensions and a CRS.")
    array_west, array_south, array_east, array_north = ndwi_array.rio.transform_bounds("EPSG:4326")
    array_doc = GISDocument(
        latitude=(array_south + array_north) / 2,
        longitude=(array_west + array_east) / 2,
        zoom=6,
        projection=MAP_PROJECTION,
    )
    display(array_doc)

In [ ]:
if data_array_layer_available:
    await array_doc.ready()
    ndwi_array_layer_id = await array_doc.add_data_array_layer(
        data_array=ndwi_array,
        name="NDWI DataArray (-1 to 1)",
        colormap_name="viridis",
        colormap_range=(-1, 1),
        opacity=1,
        zoom_to=True,
    )
    print(f"Added direct NDWI DataArray layer: {ndwi_array_layer_id}")